# Unit 12A — A vision-language-action policy: its inputs and its trainable parameters

**Your result:** the input path and the parameter selection of SmolVLA. Your functions turn camera frames, a robot state and a text instruction into the tensors the model reads, and freeze the vision-language model (VLM) so that only the action expert and its projections are trained. They match the pinned LeRobot SmolVLA implementation exactly.

Four short exercises (10–20 minutes each) and a larger experiment (30–60 minutes). These times are estimates, not measurements. Full answers are in [the reference notebook](12a-vla-inputs-reference.ipynb).

**Sources.** [SmolVLA method description](https://huggingface.co/blog/smolvla#method) (a frozen SmolVLM2 backbone, a small action expert trained with flow matching, a visual and language prefix plus the robot state), and the pinned code in `lerobot/policies/smolvla` (`modeling_smolvla.py`, `smolvlm_with_expert.py`, `processor_smolvla.py`; revision `e0d50211ef236143ae867228662b7dfaba554f02`).

**Do not mix up two things.** The SmolVLA *action expert* is a small language-model-style stack that reads the VLM's cached keys and values. It is **not** the ACT Transformer you built in Unit 8; do not assume they compute the same thing. Also keep apart the *published* policy (`smolvla-libero`, trained by others), the *prepared learner* (`smolvla-c2-5000-development`, trained in this project), and your own runs.

In [ ]:
from __future__ import annotations


from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from collections.abc import Mapping
    from collections.abc import Sequence
    from vla_lab.teaching.interfaces import ModelT
    from vla_lab.teaching.interfaces import PromptTokenizer
    from vla_lab.teaching.interfaces import VLAInputConfig
    from vla_lab.teaching.interfaces import VLAInputs
    from vla_lab.teaching.records import TrialConfig
    from vla_lab.teaching.records import TrialResult


# Supplied: workspace, offline caches, imports
from pathlib import Path
import os
import sys


def find_root() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository tree.")


WORKSPACE = find_root()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Set cache paths and offline mode before any data or model library is imported.
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS, DATA = WORKSPACE / "configs", WORKSPACE / ".local-data"
import json
import shutil
import tempfile
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

import vla_lab.teaching.smolvla as ref  # package reference: used by checks only
from vla_lab.teaching.smolvla import SmolVLAAdapter, flow_loss_fn  # Unit 12B function, package version (supplied here)
from vla_lab.teaching.act_model import load_act_samples
from vla_lab.teaching.training import optimizer_update
from lerobot.configs import FeatureType, PolicyFeature
from lerobot.policies.common.vla_utils import resize_with_pad
from lerobot.policies.smolvla import SmolVLAConfig, SmolVLAPolicy
from lerobot.processor import NewLineTaskProcessorStep, TokenizerProcessorStep, load_pretrained_policy_processors

torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("Workspace:", WORKSPACE)

In [ ]:
# Supplied: a helper that the checks use
def close(actual: torch.Tensor, expected: torch.Tensor, message: str, atol: float=1e-5) -> None:
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: what the model reads

SmolVLA reads a **prefix** and writes a chunk of actions:

- **Images.** Each camera frame (`float`, `[0, 1]`) is resized *keeping its aspect ratio* and padded on the top and left to the model size (512×512 for the published policy), then mapped to `[-1, 1]` as the vision encoder (SigLIP) expects.
- **Text.** The instruction ends with a newline, is tokenized, and is padded on the right to the longest prompt in the batch. An **attention mask** says which tokens are real; padded tokens must not be read.
- **State.** The (already normalized) robot state is zero-padded to `max_state_dim = 32`.
- **Trainable parameters.** The VLM (SigLIP + language model) is frozen. The *action expert* layers, the state projection, the action input/output projections and the action-time MLP are trained.

**Supplied here:** a small *real* SmolVLA (the pinned classes with a tiny VLM and random weights; no download, no large model load), the data reader, and the Unit 12B flow loss (package version). **You write** the four input and parameter functions.

**Real data used in this unit.** Real LIBERO `bowl_plate` frames, robot states and action chunks, read from local files (no download starts here). The recording rate is 10 Hz; nothing is resampled to the simulator's 20 Hz controller. If files are missing, the cell tells you the setup command.

In [ ]:
# Supplied: a small real SmolVLA and its tokenizer
VLM_DIR = DATA / "models/smolvlm2-500m-video-instruct"     # local tokenizer files and config; the 500M weights are NOT read here
IMAGES = ("observation.images.image", "observation.images.image2")
PROMPT = "put the bowl on the plate"


def make_tiny_smolvla(seed: int=1) -> SmolVLAPolicy:
    """A small *real* pinned SmolVLAPolicy: the same classes and code path as the 450M model,
    with a tiny VLM (width 64, 2 layers, 64x64 images). Random weights; it checks mechanics only."""
    tiny = Path(tempfile.mkdtemp(prefix="tiny_smolvlm_"))
    for path in VLM_DIR.iterdir():
        if path.is_file() and path.name != "model.safetensors":
            shutil.copy(path, tiny / path.name)
    vlm = json.loads((VLM_DIR / "config.json").read_text())
    vlm["text_config"].update(hidden_size=64, intermediate_size=128, num_attention_heads=4,
                              num_key_value_heads=2, head_dim=16, num_hidden_layers=2)
    vlm["vision_config"].update(hidden_size=32, num_attention_heads=2, num_hidden_layers=1, intermediate_size=64,
                                image_size=64, patch_size=16, max_image_size={"longest_edge": 64}, size={"longest_edge": 64})
    vlm["scale_factor"] = 2
    (tiny / "config.json").write_text(json.dumps(vlm))
    config = SmolVLAConfig(
        device="cpu", push_to_hub=False, vlm_model_name=str(tiny), load_vlm_weights=False,
        input_features={
            IMAGES[0]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            IMAGES[1]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            "observation.state": PolicyFeature(FeatureType.STATE, (8,)),
        },
        output_features={"action": PolicyFeature(FeatureType.ACTION, (7,))},
        chunk_size=10, n_action_steps=5, num_vlm_layers=2, resize_imgs_with_padding=(64, 64),
        max_state_dim=16, max_action_dim=16, prefix_length=0, num_steps=4, expert_width_multiplier=0.5,
        tokenizer_max_length=24,
    )
    torch.manual_seed(seed)
    return SmolVLAPolicy(config)


TOKENIZER = AutoTokenizer.from_pretrained(str(VLM_DIR))
POLICY = make_tiny_smolvla()
print("tiny SmolVLA parameters:", sum(p.numel() for p in POLICY.parameters()))

## Exercise 12A.1 — Camera images: resize, pad, normalize (about 15 min)

**Purpose.** Fit any camera frame into the square the vision encoder expects without distorting it, and put the pixel values in the range it was trained on.

**Contract (shapes, units, masks).**

- `resize_pad_normalize_image(image, height, width)`: `image (B, 3, h, w)` `float32` in `[0, 1]`. Return `(B, 3, height, width)` in `[-1, 1]`.
- If `(h, w)` already equals `(height, width)`, only rescale values. Otherwise: `ratio = max(w / width, h / height)`; resize bilinearly (`align_corners=False`) to `(int(h / ratio), int(w / ratio))`; **pad zeros on the top and left** until the size is `(height, width)`; then map `v → 2 * v - 1`.
- Padding is applied before the value mapping, so padded pixels end up at `-1`.
- The aspect ratio must be preserved: a wide image fills the width and the missing rows are on top.

**Method and source.** SmolVLA preprocessing (`SmolVLAPolicy.prepare_images`, `resize_with_pad` with top/left padding and `pad_value=0`, then `img * 2 - 1`), described in the [SmolVLA method section](https://huggingface.co/blog/smolvla#method). SigLIP expects values in `[-1, 1]`.

**Pointers.** Compute the ratio and the resized size first. `F.interpolate` does the resize; `F.pad(x, (left, right, top, bottom), value=0)` pads the last two axes (the order is left, right, top, bottom). Do the value mapping last.

<details><summary>Detailed hint</summary>

```python
current_height, current_width = image.shape[-2:]
if (current_height, current_width) != (height, width):
    ratio = max(current_width / width, current_height / height)
    resized = (int(current_height / ratio), int(current_width / ratio))
    image = F.interpolate(image, size=resized, mode="bilinear", align_corners=False)
    image = F.pad(image, (width - resized[1], 0, height - resized[0], 0), value=0.0)
return image * 2.0 - 1.0
```

</details>

In [ ]:
# Implement: Exercise 12A.1
def resize_pad_normalize_image(image: torch.Tensor, height: int, width: int) -> torch.Tensor:
    # (B,3,h,w) in [0,1] -> (B,3,height,width) in [-1,1]; keep the aspect ratio; pad top and left with zeros.
    raise NotImplementedError("Complete Exercise 12A.1: resize_pad_normalize_image")

In [ ]:
# Check: Exercise 12A.1
for h, w in ((64, 64), (48, 64), (64, 32), (128, 96), (256, 256)):
    image = torch.rand(2, 3, h, w)
    mine = resize_pad_normalize_image(image, 64, 64)
    expected, _ = POLICY.prepare_images({IMAGES[0]: image, IMAGES[1]: image})
    assert mine.shape == (2, 3, 64, 64), f"Expected (2, 3, 64, 64) for a {h}x{w} input."
    close(mine, expected[0], f"{h}x{w} input differs from the pinned preparation", atol=1e-6)
wide = resize_pad_normalize_image(torch.ones(1, 3, 32, 64), 64, 64)
assert (wide[:, :, :32] == -1).all() and (wide[:, :, 32:] == 1).all(), "A wide image fills the width; the padding rows are on the top and equal -1."
tall = resize_pad_normalize_image(torch.ones(1, 3, 64, 32), 64, 64)
assert (tall[..., :32] == -1).all() and (tall[..., 32:] == 1).all(), "A tall image fills the height; the padding columns are on the left."
same = resize_pad_normalize_image(torch.zeros(1, 3, 64, 64), 64, 64)
assert (same == -1).all(), "No resize when the size already fits: values only map [0, 1] -> [-1, 1]."
print("Exercise 12A.1 passed")

**Use your function.** Prepare a real 256×256 camera frame for the tiny model (64×64) and for the published model (512×512), and look at both.

In [ ]:
# Use your function: Exercise 12A.1
frames = load_act_samples(CONFIGS, DATA, "bowl_plate", "validation", count=2, chunk_size=2)     # real frames, 256x256
frame = frames[IMAGES[0]][:1]
small, large = resize_pad_normalize_image(frame, 64, 64), resize_pad_normalize_image(frame, 512, 512)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for axis, image, title in zip(axes, (frame, small, large), ("raw 256x256", "tiny model 64x64", "published model 512x512")):
    axis.imshow(((image[0] + 1) / 2 if image.min() < 0 else image[0]).permute(1, 2, 0).clamp(0, 1)); axis.set_title(title); axis.axis("off")
plt.show()
print("value range after preparation:", float(small.min()), "to", float(small.max()))

## Exercise 12A.2 — Prompts: newline, tokens and attention mask (about 15 min)

**Purpose.** Turn the instruction into token ids plus a mask that tells the model which positions are real words and which are padding.

**Contract (shapes, units, masks).**

- `tokenize_prompts(tokenizer, prompts, max_length)`: `prompts` is one string or a list of strings. `tokenizer` is a Hugging Face tokenizer.
- Append `"\n"` to every prompt that does not already end with a newline.
- Call the tokenizer with `max_length=max_length`, `truncation=True`, `padding="longest"`, `padding_side="right"`, `return_tensors="pt"`. So the length `L` is the longest prompt in the batch, at most `max_length`.
- Return `(tokens, mask)`: `tokens (B, L)` `long`, `mask (B, L)` `bool`, `True` for real tokens and `False` for padding.

**Method and source.** SmolVLA's processor pipeline: `NewLineTaskProcessorStep` then `TokenizerProcessorStep(padding='longest', padding_side='right', max_length=48)` (`processor_smolvla.py`). The mask is later used to hide padded tokens from attention.

**Pointers.** Normalize the input to a list first. Add the newline. One tokenizer call does the rest; convert the mask to `bool` at the end.

<details><summary>Detailed hint</summary>

```python
texts = [prompts] if isinstance(prompts, str) else list(prompts)
texts = [text if text.endswith("\n") else f"{text}\n" for text in texts]
encoded = tokenizer(texts, max_length=max_length, truncation=True, padding="longest",
                    padding_side="right", return_tensors="pt")
return encoded["input_ids"].long(), encoded["attention_mask"].to(torch.bool)
```

</details>

In [ ]:
# Implement: Exercise 12A.2
def tokenize_prompts(
    tokenizer: PromptTokenizer, prompts: str | Sequence[str], max_length: int
) -> tuple[torch.Tensor, torch.Tensor]:
    # Newline, tokenize with right padding to the longest prompt, return (tokens long (B,L), mask bool (B,L)).
    raise NotImplementedError("Complete Exercise 12A.2: tokenize_prompts")

In [ ]:
# Check: Exercise 12A.2
tokens, mask = tokenize_prompts(TOKENIZER, ["put the bowl on the plate", "open it\n"], 48)
assert tokens.dtype == torch.long and mask.dtype == torch.bool and tokens.shape == mask.shape and tokens.shape[0] == 2
assert mask[0].all() and mask[1].sum() < mask[0].sum(), "The shorter prompt must be right padded."
assert not mask[1][int(mask[1].sum()):].any() and mask[1][:int(mask[1].sum())].all(), "Real tokens first, then padding."
single, single_mask = tokenize_prompts(TOKENIZER, "put the bowl on the plate", 48)
assert torch.equal(single[0], tokens[0]) and single_mask.all(), "A single string is a batch of one; the newline is added."
step = TokenizerProcessorStep(tokenizer=TOKENIZER, max_length=48, padding="longest", padding_side="right")
pinned = step._tokenize_text([NewLineTaskProcessorStep().complementary_data({"task": "put the bowl on the plate"})["task"]])
assert torch.equal(single, pinned["input_ids"]), "Tokens must equal the pinned processor's tokens."
assert tokenize_prompts(TOKENIZER, "word " * 100, 12)[0].shape == (1, 12), "Truncate to max_length."
newline_twice, _ = tokenize_prompts(TOKENIZER, "open it\n", 48)
once, _ = tokenize_prompts(TOKENIZER, "open it", 48)
assert torch.equal(newline_twice, once), "Do not add a second newline."
print("Exercise 12A.2 passed")

**Use your function.** Tokenize the canonical instruction, a paraphrase and a long conflicting instruction together, and see how padding and the mask line up.

In [ ]:
# Use your function: Exercise 12A.2
prompts = ["put the bowl on the plate", "place the bowl onto the plate please", "turn on the stove and then put the bowl on the plate"]
tokens, mask = tokenize_prompts(TOKENIZER, prompts, 48)
print("token matrix:", tuple(tokens.shape), "| real tokens per prompt:", mask.sum(1).tolist())
for row in range(3):
    print(f"prompt {row}:", [TOKENIZER.decode([int(t)]) if m else "<pad>" for t, m in zip(tokens[row], mask[row])][:14])

## Exercise 12A.3 — Assemble the model inputs (about 15–20 min)

**Purpose.** Put the three kinds of input together in the form the model's prefix reads: image tensors with masks, token ids with masks, and a padded state.

**Contract (shapes, units, masks).**

- `pad_to_width(values, width)`: append zeros on the last axis until it has `width` entries; return the input unchanged if it already has `width`; raise `ValueError` if the last axis is wider than `width`.
- `prepare_vla_inputs(observation, prompts, tokenizer, config)`: `observation` has `"observation.state" (B, S)` (already normalized) and, for every key of `config.image_features` (in that order), an image `(B, 3, h, w)` in `[0, 1]`. `config.resize_imgs_with_padding = (width, height)` (note the order), `config.max_state_dim`, `config.tokenizer_max_length`.
- Return a dict: `images` (list of `(B, 3, height, width)` from Exercise 12A.1, camera order as in `config.image_features`), `img_masks` (list of `(B,)` **bool** tensors, all `True`: every camera is present), `lang_tokens`, `lang_masks` (from Exercise 12A.2 with `config.tokenizer_max_length`), `state` (`(B, config.max_state_dim)`, zero padded).
- If `prompts` is one string but the batch has `B > 1` rows, give every row the same tokens and mask (expand the single row).

**Method and source.** `SmolVLAPolicy.prepare_images`, `prepare_state` and the tokenizer steps, called with the same shapes by `VLAFlowMatching.embed_prefix` ([SmolVLA method](https://huggingface.co/blog/smolvla#method)).

**Pointers.** Call your two earlier functions. Keep the camera order of `config.image_features`. Create the masks on the same device as the images.

<details><summary>Detailed hint</summary>

```python
def pad_to_width(values, width):
    missing = width - values.shape[-1]
    if missing < 0:
        raise ValueError(f"last axis has {values.shape[-1]} entries, more than width {width}")
    return F.pad(values, (0, missing))

def prepare_vla_inputs(observation, prompts, tokenizer, config):
    width, height = config.resize_imgs_with_padding
    images = [resize_pad_normalize_image(observation[key], height, width) for key in config.image_features]
    batch = images[0].shape[0]
    tokens, token_mask = tokenize_prompts(tokenizer, prompts, config.tokenizer_max_length)
    if tokens.shape[0] == 1 and batch > 1:
        tokens, token_mask = tokens.expand(batch, -1), token_mask.expand(batch, -1)
    return {"images": images, "img_masks": [torch.ones(batch, dtype=torch.bool, device=images[0].device) for _ in images],
            "lang_tokens": tokens.to(images[0].device), "lang_masks": token_mask.to(images[0].device),
            "state": pad_to_width(observation["observation.state"], config.max_state_dim)}
```

</details>

In [ ]:
# Implement: Exercise 12A.3
def pad_to_width(values: torch.Tensor, width: int) -> torch.Tensor:
    # Zero-pad the last axis to `width`; ValueError if it is already wider.
    raise NotImplementedError("Complete Exercise 12A.3: pad_to_width")


def prepare_vla_inputs(
    observation: Mapping[str, torch.Tensor],
    prompts: str | Sequence[str],
    tokenizer: PromptTokenizer,
    config: VLAInputConfig,
) -> VLAInputs:
    # Return {"images", "img_masks", "lang_tokens", "lang_masks", "state"} from the observation and prompts.
    raise NotImplementedError("Complete Exercise 12A.3: prepare_vla_inputs")

In [ ]:
# Check: Exercise 12A.3
out = pad_to_width(torch.ones(2, 3), 5)
assert out.shape == (2, 5) and out[:, :3].eq(1).all() and out[:, 3:].eq(0).all(), "Pad with zeros on the right."
assert torch.equal(pad_to_width(torch.ones(2, 5), 5), torch.ones(2, 5))
try:
    pad_to_width(torch.ones(2, 6), 5)
except ValueError:
    pass
else:
    raise AssertionError("A wider input must raise ValueError.")

g = torch.Generator().manual_seed(0)
observation = {IMAGES[0]: torch.rand(2, 3, 48, 64, generator=g), IMAGES[1]: torch.rand(2, 3, 64, 64, generator=g),
               "observation.state": torch.randn(2, 8, generator=g)}
prompts = ["put the bowl on the plate", "open it"]
mine = prepare_vla_inputs(observation, prompts, TOKENIZER, POLICY.config)
tokens, mask = ref.tokenize_prompts(TOKENIZER, prompts, POLICY.config.tokenizer_max_length)
pinned_images, _ = POLICY.prepare_images(observation)
assert len(mine["images"]) == 2 and len(mine["img_masks"]) == 2, "One image and one mask per camera."
for a, b in zip(mine["images"], pinned_images, strict=True):
    close(a, b, "images differ from the pinned preparation", atol=1e-6)
assert all(m.dtype == torch.bool and m.shape == (2,) and m.all() for m in mine["img_masks"]), "img_masks: bool (B,) all True."
assert torch.equal(mine["lang_tokens"], tokens) and torch.equal(mine["lang_masks"], mask), "Tokens and masks from tokenize_prompts."
assert mine["state"].shape == (2, 16) and torch.equal(mine["state"][:, :8], observation["observation.state"]) and not mine["state"][:, 8:].any(), "State padded to max_state_dim with zeros."
one = prepare_vla_inputs(observation, "put the bowl on the plate", TOKENIZER, POLICY.config)
assert one["lang_tokens"].shape[0] == 2 and torch.equal(one["lang_tokens"][0], one["lang_tokens"][1]), "A single prompt is shared by the whole batch."
print("Exercise 12A.3 passed")

**Use your function.** Feed your inputs to the model and test the mask: the model's output for a prompt must not change when that prompt is padded to a longer length because another prompt in the batch is longer.

In [ ]:
# Use your function: Exercise 12A.3
policy = POLICY.eval()
adapter = SmolVLAAdapter(policy)
samples = load_act_samples(CONFIGS, DATA, "bowl_plate", "validation", count=2, chunk_size=10, image_size=64)
observation = {key: samples[key] for key in (*IMAGES, "observation.state")}
short, long_ = "put the bowl on the plate", "put the bowl on the plate and then turn on the stove next to it"
alone = prepare_vla_inputs(observation, [short, short], TOKENIZER, policy.config)
padded = prepare_vla_inputs(observation, [short, long_], TOKENIZER, policy.config)
print("prompt length alone:", alone["lang_tokens"].shape[1], "| padded to:", padded["lang_tokens"].shape[1])
noisy = torch.randn(2, 10, 16, generator=torch.Generator().manual_seed(0))
time = torch.tensor([0.5, 0.5])
with torch.no_grad():
    v_alone = adapter.velocity(alone, noisy, time)
    v_padded = adapter.velocity(padded, noisy, time)
print("output change for row 0 when its prompt is padded:", float((v_alone[0] - v_padded[0]).abs().max()))
print("output change for row 1 (a different prompt):", float((v_alone[1] - v_padded[1]).abs().max()))

## Exercise 12A.4 — Trainable parameters and the frozen VLM (about 15–20 min)

**Purpose.** Adapting a VLA should change the action expert, not the large backbone. Freeze it, give the optimizer exactly the right parameters, and keep the frozen backbone in evaluation mode when the rest trains.

**Contract (shapes, units, masks).**

- `select_trainable_parameters(policy, frozen_prefix="model.vlm_with_expert.vlm.")`: set `requires_grad = False` on every parameter whose name (from `policy.named_parameters()`) starts with `frozen_prefix`. **Do not change any other flag**: a parameter that was already frozen on purpose (for example the state projection) stays frozen. Return the list of parameters with `requires_grad=True`, each once, in `named_parameters()` order.
- `train_mode_with_frozen_vlm(policy, vlm_name="model.vlm_with_expert.vlm")`: put `policy` in training mode, then call `.eval()` on the submodule named `vlm_name` (and so on all its children). Return `policy`.
- After one optimizer update with these parameters, the VLM weights must be bit-for-bit unchanged and the expert weights must have changed.

**Method and source.** [SmolVLA](https://huggingface.co/blog/smolvla#method): the VLM is frozen and only the action expert is trained. Pinned: `SmolVLMWithExpertModel.set_requires_grad`/`train` (`train_expert_only=True`). A frozen module in training mode would still use dropout and update normalization statistics — a silent change.

**Pointers.** Loop over `named_parameters()` once for the freeze, then collect. For the mode: `model.train()` switches *everything* on; fix the backbone afterwards with `get_submodule(name).eval()`.

<details><summary>Detailed hint</summary>

```python
def select_trainable_parameters(policy, frozen_prefix="model.vlm_with_expert.vlm."):
    for name, parameter in policy.named_parameters():
        if name.startswith(frozen_prefix):
            parameter.requires_grad_(False)
    return [parameter for _, parameter in policy.named_parameters() if parameter.requires_grad]

def train_mode_with_frozen_vlm(policy, vlm_name="model.vlm_with_expert.vlm"):
    policy.train()
    policy.get_submodule(vlm_name).eval()
    return policy
```

</details>

In [ ]:
# Implement: Exercise 12A.4
def select_trainable_parameters(
    policy: ModelT, frozen_prefix: str = "model.vlm_with_expert.vlm."
) -> list[torch.nn.Parameter]:
    # Freeze every parameter under frozen_prefix; leave all other flags alone; return the trainable parameters.
    raise NotImplementedError("Complete Exercise 12A.4: select_trainable_parameters")


def train_mode_with_frozen_vlm(
    policy: ModelT, vlm_name: str = "model.vlm_with_expert.vlm"
) -> ModelT:
    # policy.train(), then keep the VLM submodule in eval mode. Return policy.
    raise NotImplementedError("Complete Exercise 12A.4: train_mode_with_frozen_vlm")

In [ ]:
# Check: Exercise 12A.4
model = make_tiny_smolvla(seed=3)
for parameter in model.parameters():                      # something unfroze everything
    parameter.requires_grad_(True)
trainable = select_trainable_parameters(model)
owner = {id(p): name for name, p in model.named_parameters()}
assert len({id(p) for p in trainable}) == len(trainable), "Return each parameter once."
assert not any(owner[id(p)].startswith("model.vlm_with_expert.vlm.") for p in trainable), "No VLM parameter may be returned."
assert all(not p.requires_grad for n, p in model.named_parameters() if n.startswith("model.vlm_with_expert.vlm.")), "Every VLM parameter must be frozen."
names = [owner[id(p)] for p in trainable]
assert any(n.startswith("model.vlm_with_expert.lm_expert.") for n in names) and any(n.startswith("model.action_out_proj") for n in names), "Expert and projections stay trainable."
model.model.state_proj.requires_grad_(False)               # a deliberate freeze must survive
assert not any(owner[id(p)].startswith("model.state_proj") for p in select_trainable_parameters(model)), "Do not unfreeze parameters outside the prefix."

model.model.vlm_with_expert.vlm.train()                    # something switched the backbone on
assert train_mode_with_frozen_vlm(model) is model
assert model.training and model.model.action_out_proj.training and model.model.vlm_with_expert.lm_expert.training, "The rest of the model must be in training mode."
assert not any(m.training for m in model.model.vlm_with_expert.vlm.modules()), "Every VLM submodule must be in eval mode."

# One real update: the VLM stays bit-for-bit identical, the expert changes.
model = make_tiny_smolvla(seed=3)
for parameter in model.parameters():
    parameter.requires_grad_(True)
params = select_trainable_parameters(model)
optimizer = torch.optim.AdamW(params, lr=1e-2)
before_vlm = {n: p.detach().clone() for n, p in model.model.vlm_with_expert.vlm.named_parameters()}
before_head = model.model.action_out_proj.weight.detach().clone()
adapter = SmolVLAAdapter(model, train_mode_fn=train_mode_with_frozen_vlm)
g = torch.Generator().manual_seed(0)
batch = {IMAGES[0]: torch.rand(2, 3, 64, 64, generator=g), IMAGES[1]: torch.rand(2, 3, 64, 64, generator=g),
         "observation.state": torch.randn(2, 8, generator=g),
         "observation.language.tokens": torch.randint(5, 100, (2, 6)), "observation.language.attention_mask": torch.ones(2, 6, dtype=torch.bool),
         "action": torch.randn(2, 10, 7, generator=g), "action_is_pad": torch.zeros(2, 10, dtype=torch.bool)}
optimizer_update(adapter, batch, optimizer, flow_loss_fn, noise=torch.randn(2, 10, 16, generator=g), time=torch.tensor([0.3, 0.7]))
assert all(torch.equal(before_vlm[n], p) for n, p in model.model.vlm_with_expert.vlm.named_parameters()), "The frozen VLM must not change."
assert not torch.equal(before_head, model.model.action_out_proj.weight), "The expert side must change."
print("Exercise 12A.4 passed")

**Use your function.** Count where the parameters of the small model are, from your selection. The same grouping is used for the 450M prepared learner.

In [ ]:
# Use your function: Exercise 12A.4
model = make_tiny_smolvla(seed=3)
total = sum(p.numel() for p in model.parameters())
trainable = select_trainable_parameters(model)
groups = {"VLM (frozen)": "model.vlm_with_expert.vlm.", "action expert": "model.vlm_with_expert.lm_expert.", "state projection": "model.state_proj.",
          "action in/out projections": ("model.action_in_proj.", "model.action_out_proj."), "action-time MLP": ("model.action_time_mlp_in.", "model.action_time_mlp_out.")}
for label, prefix in groups.items():
    count = sum(p.numel() for n, p in model.named_parameters() if n.startswith(prefix))
    trainable_count = sum(p.numel() for n, p in model.named_parameters() if n.startswith(prefix) and p.requires_grad)
    print(f"{label:<26} {count:>9,} parameters, {trainable_count:>9,} trainable")
print(f"total {total:,} | selected for the optimizer {sum(p.numel() for p in trainable):,}")

### Your inputs against the real published configuration

The published policy (`smolvla-libero`) uses 512×512 images, a 48-token prompt limit and 32-wide state and action vectors. This check builds the *real* preprocessing pipeline of that policy from its saved files (configuration, tokenizer, normalization; no 450M weights are loaded) and compares your `prepare_vla_inputs` with it on real camera frames. It reads local files only. If they are missing, the cell says so.

In [ ]:
RUN_REAL_CONFIG_CHECK = True

if RUN_REAL_CONFIG_CHECK:
    model_dir = DATA / "models/smolvla-libero"
    if not (model_dir / "policy_preprocessor.json").is_file():
        print("Published SmolVLA files not found; skipping. Install them with `just data-models` (this downloads; run it outside this notebook).")
    else:
        real_config = SmolVLAConfig.from_pretrained(str(model_dir))
        pipeline, _ = load_pretrained_policy_processors(
            str(model_dir), preprocessor_overrides={"tokenizer_processor": {"tokenizer_name": str(VLM_DIR)}, "device_processor": {"device": "cpu"}})
        real_frames = load_act_samples(CONFIGS, DATA, "bowl_plate", "validation", count=2, chunk_size=2)
        raw = {IMAGES[0]: real_frames[IMAGES[0]][0], IMAGES[1]: real_frames[IMAGES[1]][0],
               "observation.state": real_frames["observation.state"][0], "task": PROMPT}
        pinned_batch = pipeline(raw)                                   # the real pipeline: tokens, masks, normalized state
        mine = prepare_vla_inputs({k: v for k, v in pinned_batch.items() if k.startswith("observation.")}, PROMPT, TOKENIZER, real_config)
        print("image size:", tuple(mine["images"][0].shape), "| state width:", mine["state"].shape[-1], "| prompt tokens:", mine["lang_tokens"].shape[1])
        for a, key in zip(mine["images"], real_config.image_features, strict=True):
            expected = resize_with_pad(pinned_batch[key], 512, 512, pad_value=0) * 2 - 1
            assert (a - expected).abs().max() < 1e-6, f"{key} differs from the pinned resize/pad/scale"
        assert torch.equal(mine["lang_tokens"], pinned_batch["observation.language.tokens"]), "tokens differ from the real pipeline"
        assert torch.equal(mine["lang_masks"], pinned_batch["observation.language.attention_mask"]), "mask differs from the real pipeline"
        assert mine["state"].shape == (1, 32) and torch.equal(mine["state"][:, :8], pinned_batch["observation.state"])
        print("Your inputs equal the real published preprocessing on real frames.")
else:
    print("Real-configuration check is off.")

## Experiment — which parameters should be trained? (30–60 min)

**Question.** With the same updates, how do three adaptation settings differ in trainable parameters, held-out flow loss, and whether the frozen backbone really stays fixed?

**Setup (real data, small real model).** 128 training and 64 validation windows, evenly spread over the frozen `bowl_plate` train and validation episodes: both camera frames (box-filtered to 64×64), the standardized state, 10 standardized actions, padding where the chunk passes the episode end, and the instruction `put the bowl on the plate`. The model is the small real SmolVLA from the supplied cell (random weights: this is a mechanics experiment, not a robot policy). **Your functions** `resize_pad_normalize_image`, `tokenize_prompts`, `pad_to_width`, `prepare_vla_inputs`, `select_trainable_parameters` and `train_mode_with_frozen_vlm` are all in the real update: the adapter's `input_fn` builds the inputs with them, and `train_mode_fn` keeps the VLM frozen. The loss is the Unit 12B flow loss (package version) through the training unit's `optimizer_update`.

**Fixed for every trial.** Weight seed `1`; batch seed `1`; **300 updates**, batch 16, AdamW `lr=1e-3`; the noise and the times of every update come from the seeded generator (times uniform in `(0.001, 0.999)`); the same validation noise (seed `7`) at times `0.2, 0.5, 0.8` for every trial.

**Allowed changes (per trial).** `trainable`: `expert_only` (your selection: VLM frozen, expert and projections trained), `expert_and_vlm_text` (also unfreeze the VLM language model), `projections_only` (freeze the expert layers as well).

**Measures.** `trainable_params`; `val_flow_loss`; `vlm_drift`: the largest absolute change of any VLM weight (must be exactly `0` when the VLM is frozen); `seconds`.

**Prediction first.** In the next cell: which setting reaches the lowest validation loss, and which keeps `vlm_drift` at 0?

**Keep failed trials.** Every trial is appended to `TRIALS`. A setting that is no better than the cheaper one is a valid result.

In [ ]:
# Larger training experiment: run only when you opt in.
RUN_EXPERIMENT = False

In [ ]:
if RUN_EXPERIMENT:
    PREDICTION = "TODO: replace this text with your prediction."
    print(PREDICTION)

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: trial table helper
    import time
    import pandas as pd

    def show_trials(trials: Sequence[TrialResult], columns: Sequence[str]) -> pd.DataFrame:
        """Print every trial that was run, including failed ones, and return a table."""
        table = pd.DataFrame([{**t["config"], **{c: t.get(c) for c in columns}, "status": t["status"]} for t in trials])
        print(table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        return table

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: real windows for the small model (64x64 frames, 10-step chunks)
    TRAIN = load_act_samples(CONFIGS, DATA, "bowl_plate", "train", count=128, chunk_size=10, image_size=64)
    VALID = load_act_samples(CONFIGS, DATA, "bowl_plate", "validation", count=64, chunk_size=10, image_size=64)
    print("train windows:", len(TRAIN["action"]), "| validation windows:", len(VALID["action"]), "| instruction:", repr(PROMPT))

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: harness. Your input and parameter functions are in the real update.
    STEPS, BATCH, LR, SEED_WEIGHTS, SEED_BATCH, SEED_NOISE = 300, 16, 1e-3, 1, 1, 7
    BASELINE = {"trainable": "expert_only"}
    TRIALS = []


    def run_trial(config: TrialConfig) -> dict[str, float | int]:
        started = time.perf_counter()
        policy = make_tiny_smolvla(seed=SEED_WEIGHTS)
        for parameter in policy.parameters():
            parameter.requires_grad_(True)
        selected = select_trainable_parameters(policy)                      # YOUR function freezes the VLM
        vlm = policy.model.vlm_with_expert.vlm
        if config["trainable"] == "expert_and_vlm_text":
            for parameter in vlm.model.text_model.parameters():
                parameter.requires_grad_(True)
        if config["trainable"] == "projections_only":
            for parameter in policy.model.vlm_with_expert.lm_expert.parameters():
                parameter.requires_grad_(False)
        params = [p for p in policy.parameters() if p.requires_grad]
        before = {name: p.detach().clone() for name, p in vlm.named_parameters()}

        def input_fn(adapter: SmolVLAAdapter, raw: Mapping[str, torch.Tensor]) -> VLAInputs:                                         # YOUR input functions build every batch
            observation = {key: value for key, value in raw.items() if key.startswith("observation.")}
            inputs = prepare_vla_inputs(observation, PROMPT, TOKENIZER, policy.config)
            if "action" in raw:
                inputs["actions"] = pad_to_width(raw["action"], policy.config.max_action_dim)
            return inputs

        adapter = SmolVLAAdapter(policy, train_mode_fn=train_mode_with_frozen_vlm, input_fn=input_fn)
        optimizer = torch.optim.AdamW(params, lr=LR)
        draws = torch.Generator().manual_seed(SEED_BATCH)
        for _ in range(STEPS):
            pick = torch.randint(0, len(TRAIN["action"]), (BATCH,), generator=draws)
            batch = {key: value[pick] for key, value in TRAIN.items()}
            noise = torch.randn(BATCH, 10, policy.config.max_action_dim, generator=draws)
            time_ = torch.rand(BATCH, generator=draws) * 0.998 + 0.001
            optimizer_update(adapter, batch, optimizer, flow_loss_fn, noise=noise, time=time_)
        adapter.eval()
        with torch.no_grad():
            losses = []
            for t in (0.2, 0.5, 0.8):
                noise = torch.randn(len(VALID["action"]), 10, policy.config.max_action_dim, generator=torch.Generator().manual_seed(SEED_NOISE))
                loss, _ = flow_loss_fn(adapter, VALID, noise=noise, time=torch.full((len(noise),), t))
                losses.append(loss.item())
        return {
            "trainable_params": sum(p.numel() for p in params),
            "val_flow_loss": sum(losses) / len(losses),
            "vlm_drift": max((p - before[name]).abs().max().item() for name, p in vlm.named_parameters()),
            "seconds": time.perf_counter() - started,
        }


    def record(config: TrialConfig) -> None:
        config = {**BASELINE, **config}
        try:
            TRIALS.append({"config": config, "status": "ok", **run_trial(config)})
        except Exception as error:  # a failed trial is evidence too
            TRIALS.append({"config": config, "status": f"failed: {type(error).__name__}: {error}"})


    record(BASELINE)
    COLUMNS = ["trainable_params", "val_flow_loss", "vlm_drift", "seconds"]
    show_trials(TRIALS, COLUMNS);

**Your trials.** Add at least two trials. Budget: 300 updates per trial.

In [ ]:
if RUN_EXPERIMENT:
    # Implement: choose your trials.
    MY_TRIALS = [
        # {"trainable": "projections_only"},
    ]
    for trial in MY_TRIALS:
        record(trial)
    show_trials(TRIALS, COLUMNS);

In [ ]:
if RUN_EXPERIMENT:
    # Check: the protocol was followed.
    assert TRIALS[0]["config"] == BASELINE, "The first trial must be the baseline."
    assert len(TRIALS) >= 3, "Run the baseline and at least two trials of your own."
    assert all(t["config"]["trainable"] in {"expert_only", "expert_and_vlm_text", "projections_only"} for t in TRIALS)
    ok = [t for t in TRIALS if t["status"] == "ok"]
    assert all(torch.isfinite(torch.tensor(t["val_flow_loss"])) for t in ok), "val_flow_loss must be finite."
    frozen = [t for t in ok if t["config"]["trainable"] != "expert_and_vlm_text"]
    assert all(t["vlm_drift"] == 0.0 for t in frozen), "A frozen VLM must not drift: check select_trainable_parameters and train_mode_with_frozen_vlm."
    print(f"{len(TRIALS)} trials kept ({len(ok)} ok). Budget: {STEPS} updates, batch {BATCH}, seeds {SEED_WEIGHTS}/{SEED_BATCH}/{SEED_NOISE}.")

In [ ]:
if RUN_EXPERIMENT:
    # Use your function: parameters against held-out loss, every trial shown
    ok = [t for t in TRIALS if t["status"] == "ok"]
    labels = [t["config"]["trainable"] for t in ok]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].bar(range(len(ok)), [t["val_flow_loss"] for t in ok]); axes[0].set_title("val_flow_loss")
    axes[1].bar(range(len(ok)), [t["trainable_params"] for t in ok]); axes[1].set_title("trainable parameters"); axes[1].set_yscale("log")
    for axis in axes:
        axis.set_xticks(range(len(ok))); axis.set_xticklabels(labels, rotation=20, ha="right")
    plt.tight_layout(); plt.show()

**Explain (2–4 sentences, edit this cell).** Which setting gave the best held-out loss for its cost? Which setting broke the "frozen VLM" rule, and what did `vlm_drift` show? This VLM has random weights and 128 training windows: what can you *not* conclude about the real 450M model?

*Optional extension:* rerun the baseline with another `SEED_WEIGHTS` and keep both rows.

## Opt-in: the prepared learner at real size (CPU or GPU, one model)

This loads the **prepared learner** (`smolvla-c2-5000-development`, 450M parameters, trained in this project) and its processors from local files (hash-checked, no download), then:

1. runs **your** `select_trainable_parameters` on it and compares the group counts with the project's own parameter audit;
2. builds one real batch (two camera frames, state, instruction) with the real pipeline, and compares your `prepare_vla_inputs` with the pinned path at the 512×512 size;
3. computes the velocity from both input paths at fixed noise and time (`SmolVLAAdapter.velocity`) and compares them.

It makes no update and writes nothing. The code selects CUDA when available and otherwise uses CPU. It loads one model. Set `RUN_REAL_SMOLVLA = True` to run it.

In [ ]:
RUN_REAL_SMOLVLA = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if RUN_REAL_SMOLVLA:
    from torch.utils.data import default_collate
    from vla_lab.policy.smolvla_learner import audit_learner_parameter_groups, load_learner_bundle
    from vla_lab.teaching.act_model import open_libero_dataset

    checkpoint = DATA / "runs/smolvla-c2-5000-development/checkpoint-005000"
    if not (checkpoint / "model.safetensors").is_file() or not VLM_DIR.joinpath("model.safetensors").is_file():
        raise FileNotFoundError("Prepared learner or VLM weights missing (see manifests/smolvla_reference.yaml).")
    learner, _, _, manifest = load_learner_bundle(checkpoint, VLM_DIR, device=DEVICE)      # weights and hashes are checked
    preprocessor, _ = load_pretrained_policy_processors(
        str(checkpoint), preprocessor_overrides={"tokenizer_processor": {"tokenizer_name": str(VLM_DIR)}, "device_processor": {"device": DEVICE}})
    for parameter in learner.parameters():
        parameter.requires_grad_(True)
    trainable = select_trainable_parameters(learner)
    groups = audit_learner_parameter_groups(learner)                       # raises if a flag breaks the project's rule
    counted = sum(p.numel() for p in trainable)
    total = sum(p.numel() for p in learner.parameters())
    audited = sum(group["trainable_parameters"] for group in groups.values())
    print(f"your selection: {counted:,} trainable of {total:,} parameters | project audit: {audited:,} | frozen VLM: {total - counted:,}")
    assert counted == audited, "your selection differs from the project's parameter audit"
    vlm_params = sum(p.numel() for n, p in learner.named_parameters() if n.startswith("model.vlm_with_expert.vlm."))
    assert total - counted >= vlm_params, "every VLM parameter must be frozen"
    dataset = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "validation", {"action": [i / 10 for i in range(learner.config.chunk_size)]})
    batch = preprocessor(default_collate([dataset[5]]))
    adapter = SmolVLAAdapter(learner.eval())
    pinned_inputs = adapter.pinned_inputs(batch)
    observation = {k: v for k, v in batch.items() if k.startswith("observation.") and "language" not in k}
    mine = prepare_vla_inputs(observation, PROMPT, AutoTokenizer.from_pretrained(str(VLM_DIR)), learner.config)
    same = all(torch.equal(a, b) for a, b in zip(mine["images"], pinned_inputs["images"], strict=True)) and torch.equal(mine["lang_tokens"], pinned_inputs["lang_tokens"]) \
        and torch.equal(mine["lang_masks"], pinned_inputs["lang_masks"]) and torch.equal(mine["state"], pinned_inputs["state"])
    print("your inputs equal the pinned inputs at real size:", same)
    noise = torch.randn(1, learner.config.chunk_size, learner.config.max_action_dim, generator=torch.Generator().manual_seed(0)).to(DEVICE)
    time_ = torch.tensor([0.5], device=DEVICE)
    with torch.no_grad():
        difference = (adapter.velocity(mine, noise, time_) - adapter.velocity(pinned_inputs, noise, time_)).abs().max().item()
    print("largest velocity difference between the two input paths:", difference)
else:
    print("Real SmolVLA check is off. Set RUN_REAL_SMOLVLA=True to run it (loads one 450M model).")

## What you built

- `resize_pad_normalize_image`, `tokenize_prompts`, `pad_to_width`, `prepare_vla_inputs`, `select_trainable_parameters` and `train_mode_with_frozen_vlm`: the input path and parameter selection of SmolVLA, equal to the pinned implementation, with a mask that keeps padded prompt tokens from changing the output.
- A fixed-budget comparison of adaptation settings with every trial kept, and a check that a frozen backbone does not move.

**Next:** Unit 12B writes the flow-matching objective and the Euler sampler, and puts them into the same real model.